# Descarga de velas (klines) de Binance

Descarga velas spot desde `data.binance.vision` (público, sin API key), verifica el SHA-256 de cada archivo y guarda un CSV con el rango pedido.

- Meses cerrados: archivo mensual. Meses aún no publicados como mensual: archivos diarios.
- Desde 2025 los timestamps de spot vienen en microsegundos; se detecta por magnitud.
- Tiempos en UTC.

## 1. Imports

In [ ]:
import hashlib
import io
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

## 1.1 [CONFIG]

In [ ]:
RUTA         = Path(r"C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\data\reales\raw\cripto_binance")
PAR          = "BTCUSDT"
INTERVALO    = "5m"            # 1m, 3m, 5m, 15m, 30m, 1h, 2h, 4h, 6h, 8h, 12h, 1d
FECHA_INICIO = "2023-01-01"    # inclusive, UTC
FECHA_FIN    = "2026-08-31"    # inclusive, UTC

## 2. Funciones

In [ ]:
BASE = "https://data.binance.vision/data/spot"
COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
        "quote_volume", "n_trades", "taker_buy_base", "taker_buy_quote", "ignore"]


def _bajar(url):
    with urllib.request.urlopen(url, timeout=60) as r:
        return r.read()


def obtener_zip(frecuencia, etiqueta, cache):
    """Zip verificado (bytes) o None si no existe. frecuencia: 'monthly' | 'daily'."""
    nombre = f"{PAR}-{INTERVALO}-{etiqueta}.zip"
    ruta = cache / frecuencia / nombre
    if ruta.exists():
        return ruta.read_bytes()
    url = f"{BASE}/{frecuencia}/klines/{PAR}/{INTERVALO}/{nombre}"
    try:
        datos = _bajar(url)
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return None
        raise
    sha = _bajar(url + ".CHECKSUM").decode().split()[0]
    if hashlib.sha256(datos).hexdigest() != sha:
        raise ValueError(f"checksum no coincide: {nombre}")
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_bytes(datos)
    return datos


def leer_zip(datos):
    with zipfile.ZipFile(io.BytesIO(datos)) as z:
        df = pd.read_csv(z.open(z.namelist()[0]), header=None, names=COLS)
    if not str(df.iloc[0, 0]).isdigit():   # algunos archivos traen encabezado
        df = df.iloc[1:].astype({c: float for c in COLS})
    for c in ("open_time", "close_time"):
        unidad = "us" if df[c].iloc[0] > 1e14 else "ms"
        df[c] = pd.to_datetime(df[c].astype("int64"), unit=unidad, utc=True)
    return df.drop(columns="ignore")

## 3. Descarga

In [ ]:
inicio = pd.Timestamp(FECHA_INICIO, tz="UTC")
fin    = pd.Timestamp(FECHA_FIN, tz="UTC")
assert inicio <= fin, "FECHA_INICIO debe ser <= FECHA_FIN"

cache = RUTA / PAR / INTERVALO / "zip"
partes, faltantes = [], []

for mes in pd.period_range(inicio.tz_localize(None), fin.tz_localize(None), freq="M"):
    datos = obtener_zip("monthly", mes.strftime("%Y-%m"), cache)
    if datos is not None:
        partes.append(leer_zip(datos))
        print("mensual", mes)
        continue
    # mes aún no publicado como mensual: días sueltos
    dias = pd.date_range(max(mes.start_time, inicio.tz_localize(None)),
                         min(mes.end_time, fin.tz_localize(None)), freq="D")
    for dia in dias:
        datos = obtener_zip("daily", dia.strftime("%Y-%m-%d"), cache)
        if datos is None:
            faltantes.append(dia.date())
        else:
            partes.append(leer_zip(datos))
    print("diario ", mes, f"({len(dias)} días)")

assert partes, "no se descargó nada: revisar PAR, INTERVALO y fechas"
velas = (pd.concat(partes)
         .drop_duplicates("open_time")
         .sort_values("open_time")
         .loc[lambda d: (d["open_time"] >= inicio) & (d["open_time"] < fin + pd.Timedelta(days=1))]
         .reset_index(drop=True))
print("días sin archivo:", faltantes if faltantes else "ninguno")

## 4. Guardado y resumen

In [ ]:
archivo = RUTA / f"{PAR}_{INTERVALO}_{inicio:%Y%m%d}_{fin:%Y%m%d}.csv"
velas.to_csv(archivo, index=False)

por_dia = velas.groupby(velas["open_time"].dt.floor("D")).size()
esperado = por_dia.mode().iloc[0]
print(archivo.resolve())
print(f"velas: {len(velas):,}   días: {len(por_dia)}   velas/día esperadas: {esperado}")
print(f"días incompletos: {(por_dia < esperado).sum()}")
velas.head()